<a href="https://colab.research.google.com/github/NabaaNabeeh/sms-spam-misspellings/blob/main/SMS_Spam_Implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SMS Spam Detection under Misspellings
**Group 1**

Run the notebook from top to bottom in Google Colab (Runtime > Run all). It loads the UCI SMS Spam Collection, creates a shared training, development and test split, builds a misspelled copy of the test set, and trains Method 1 (TF-IDF + Logistic Regression) and Method 2 (all-MiniLM-L6-v2 sentence embeddings + Logistic Regression). Settings are selected on development data, and the Try it section accepts new SMS input. The test sets are reserved for the final comparison.

## Setup

In [1]:
from pathlib import Path
from io import BytesIO
from urllib.request import urlopen
from urllib.error import URLError
from zipfile import ZipFile

import random
import joblib
import pandas as pd
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
import sentence_transformers
from sentence_transformers import SentenceTransformer

SEED = 42
OUTPUT = Path("outputs")
OUTPUT.mkdir(exist_ok=True)
print(f"Verified environment: pandas {pd.__version__}, scikit-learn {sklearn.__version__}, joblib {joblib.__version__}, sentence-transformers {sentence_transformers.__version__}")

Verified environment: pandas 2.2.3, scikit-learn 1.6.1, joblib 1.6.0, sentence-transformers 5.7.0


## Data
Keep the message text for both methods. Use a separate key that ignores case and whitespace to identify duplicates. Remove duplicate groups with conflicting labels, then keep one row per remaining key. IDs come from the source row order.

In [2]:
# Try the original UCI source first; if it fails (e.g. an expired SSL certificate),
# fall back to the team's GitHub copy of the same file.
DATA_FILE = Path("data/SMSSpamCollection")
UCI_URL = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
GITHUB_URL = "https://raw.githubusercontent.com/NabaaNabeeh/sms-spam-misspellings/main/data/SMSSpamCollection"

if not DATA_FILE.exists():
    DATA_FILE.parent.mkdir(exist_ok=True)
    try:
        with urlopen(UCI_URL, timeout=45) as response:
            with ZipFile(BytesIO(response.read())) as archive:
                DATA_FILE.write_bytes(archive.read("SMSSpamCollection"))
        print("Data source: UCI")
    except (URLError, TimeoutError) as error:
        print(f"UCI download failed ({error}). Data source: team GitHub copy")
        with urlopen(GITHUB_URL, timeout=45) as response:
            DATA_FILE.write_bytes(response.read())

lines = DATA_FILE.read_text(encoding="utf-8").splitlines()

records = []
for number, line in enumerate(lines, start=1):
    if line.strip():
        label, separator, text = line.partition("\t")
        if not separator:
            raise ValueError(f"Missing label separator on source line {number}")
        records.append((f"sms_{number:05d}", text, label.strip().lower()))

raw = pd.DataFrame(records, columns=["message_id", "text", "label"])
assert set(raw["label"]) == {"ham", "spam"}
valid = raw.dropna(subset=["text", "label"]).copy()
valid = valid.loc[valid["text"].str.strip().ne("")].copy()
valid["duplicate_key"] = valid["text"].map(
    lambda text: " ".join(text.casefold().split())
)
label_counts = valid.groupby("duplicate_key")["label"].transform("nunique")
conflicting = int(label_counts.gt(1).sum())
consistent = valid.loc[label_counts.eq(1)]
clean = consistent.drop_duplicates("duplicate_key").copy()

print(f"Blank or missing rows removed: {len(raw) - len(valid)}")
print(f"Rows removed for conflicting labels: {conflicting}")
print(f"Duplicate rows removed: {len(consistent) - len(clean)}")

Data source: UCI
Blank or missing rows removed: 0
Rows removed for conflicting labels: 0
Duplicate rows removed: 415


In [3]:
train, holdout = train_test_split(
    clean, test_size=0.30, random_state=SEED, stratify=clean["label"]
)
dev, test = train_test_split(
    holdout, test_size=0.50, random_state=SEED, stratify=holdout["label"]
)

splits = {"train": train, "dev": dev, "test": test}
all_rows = pd.concat(splits.values(), ignore_index=True)
assert len(all_rows) == len(clean)
assert all_rows["message_id"].is_unique
assert all_rows["duplicate_key"].is_unique

for name, frame in splits.items():
    frame[["message_id", "text", "label"]].to_csv(OUTPUT / f"{name}.csv", index=False)

summary = pd.DataFrame({
    name: frame["label"].value_counts()
    for name, frame in {"Raw": raw, "Cleaned": clean, **splits}.items()
}).T[["ham", "spam"]]
summary["total"] = summary.sum(axis=1)
summary.to_csv(OUTPUT / "data_summary.csv")
print(summary.to_string())

label     ham  spam  total
Raw      4827   747   5574
Cleaned  4517   642   5159
train    3162   449   3611
dev       677    97    774
test      678    96    774


### Misspelled test set
A misspelled copy of the test set is created to address the experimental question. In each spam message, approximately 30% of the words containing the letters o, e, i, a or s are disguised through character substitution (for example, "free" becomes "fr33"). The replacement characters resemble the original letters, so a reader still understands the word while its exact spelling changes; this tests whether each method still recognises the message as spam. Ham messages remain unchanged. Words are selected at random with a fixed seed, so the file is identical on every run. Neither method is trained on this set.

In [4]:
# Misspelled Test Set

import random

random.seed(SEED)

SUBSTITUTIONS = {
    "a": "4",
    "e": "3",
    "i": "1",
    "o": "0",
    "s": "5"
}

def disguise_word(word):
    chars = list(word)

    for i, ch in enumerate(chars):
        if ch.lower() in SUBSTITUTIONS:
            chars[i] = SUBSTITUTIONS[ch.lower()]

    return "".join(chars)

def add_misspellings(text, ratio=0.30):
    words = text.split()

    candidates = [
        i for i, word in enumerate(words)
        if any(c.lower() in SUBSTITUTIONS for c in word)
    ]

    if not candidates:
        return text

    n_words = max(1, round(len(candidates) * ratio))

    selected = random.sample(
        candidates,
        min(n_words, len(candidates))
    )

    for idx in selected:
        words[idx] = disguise_word(words[idx])

    return " ".join(words)

# Create misspelled copy
misspelled_test = test.copy()

spam_mask = misspelled_test["label"] == "spam"

misspelled_test.loc[spam_mask, "text"] = (
    misspelled_test.loc[spam_mask, "text"]
    .apply(add_misspellings)
)

print("Original example:")
print(test.loc[spam_mask, "text"].iloc[0])

print("\nMisspelled example:")
print(misspelled_test.loc[spam_mask, "text"].iloc[0])

Original example:
WINNER!! As a valued network customer you have been selected to receivea £900 prize reward! To claim call 09061701461. Claim code KL341. Valid 12 hours only.

Misspelled example:
W1NN3R!! As a v4lu3d n3tw0rk customer you h4v3 b33n selected to receivea £900 prize reward! To claim call 09061701461. Cl41m code KL341. Valid 12 h0ur5 only.


## Method 1
Fit TF-IDF and Logistic Regression on training data. Compare two regularization settings using development macro F1, which gives equal weight to Ham and Spam. If the scores tie, retain the first setting.

In [5]:
models = {}
scores = []
for c in (1.0, 4.0):
    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
        ("classifier", LogisticRegression(C=c, max_iter=1000, random_state=SEED)),
    ])
    pipeline.fit(train["text"], train["label"])
    predictions = pipeline.predict(dev["text"])
    models[c] = pipeline
    scores.append({
        "C": c,
        "accuracy": accuracy_score(dev["label"], predictions),
        "macro_f1": f1_score(dev["label"], predictions, average="macro"),
    })

best = max(scores, key=lambda row: row["macro_f1"])
model = models[best["C"]]
print(pd.DataFrame(scores).round(4).to_string(index=False))
print(f"Selected C: {best['C']}")
pd.DataFrame(scores).to_csv(OUTPUT / "development_scores.csv", index=False)
joblib.dump(model, OUTPUT / "method1.joblib")

  C  accuracy  macro_f1
1.0    0.9496    0.8602
4.0    0.9793    0.9492
Selected C: 4.0


['outputs/method1.joblib']

## Method 2
Encode each message with the frozen pretrained model all-MiniLM-L6-v2 (sentence-transformers), then train Logistic Regression on the embeddings. The C values, classifier settings and selection rule match Method 1, so the only difference is the text representation.

In [6]:
# Load the pretrained encoder. Its weights stay frozen; only the classifier is trained later.
# Running on CPU keeps the embeddings identical across sessions.
encoder = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")

def embed(texts):
    """Convert messages into 384-dimensional sentence embeddings."""
    return encoder.encode(list(texts), batch_size=64, show_progress_bar=False)

X_train = embed(train["text"])
X_dev = embed(dev["text"])
print(f"Train embeddings: {X_train.shape}, Dev embeddings: {X_dev.shape}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Train embeddings: (3611, 384), Dev embeddings: (774, 384)


In [7]:
# Train Logistic Regression on the embeddings, using the same C values and settings as Method 1.
models2 = {}
scores2 = []
for c in (1.0, 4.0):
    classifier = LogisticRegression(C=c, max_iter=1000, random_state=SEED)
    classifier.fit(X_train, train["label"])
    predictions = classifier.predict(X_dev)
    models2[c] = classifier
    scores2.append({
        "C": c,
        "accuracy": accuracy_score(dev["label"], predictions),
        "macro_f1": f1_score(dev["label"], predictions, average="macro"),
    })

# Keep the C with the highest development macro F1 (on a tie, the first one wins, as in Method 1).
best2 = max(scores2, key=lambda row: row["macro_f1"])
model2 = models2[best2["C"]]
print(pd.DataFrame(scores2).round(4).to_string(index=False))
print(f"Selected C: {best2['C']}")
pd.DataFrame(scores2).to_csv(OUTPUT / "method2_development_scores.csv", index=False)
joblib.dump(model2, OUTPUT / "method2_classifier.joblib")

  C  accuracy  macro_f1
1.0    0.9755    0.9406
4.0    0.9858    0.9662
Selected C: 4.0


['outputs/method2_classifier.joblib']

In [8]:
def predict_method2(message):
    """Predict 'spam' or 'ham' for one message using Method 2 (no retraining)."""
    return model2.predict(embed([message]))[0]

# Quick check with the two examples from the project guidelines.
for example in [
    "You won a cash prize. Pay a fee now to claim it",
    "Please bring your notebook to tomorrow's lab",
]:
    print(f"{predict_method2(example)}: {example}")

spam: You won a cash prize. Pay a fee now to claim it
ham: Please bring your notebook to tomorrow's lab


## Results and Evaluation

The two methods were evaluated on the same test set using accuracy and macro F1. Then, both methods were evaluated on a modified version containing misspelled spam messages.



In [9]:
# Final Evaluation on Original Test Set

from sklearn.metrics import accuracy_score, f1_score

# Method 1 predictions
m1_test_pred = model.predict(test["text"])

# Method 2 predictions
X_test = embed(test["text"])
m2_test_pred = model2.predict(X_test)

results_original = pd.DataFrame({
    "Method": [
        "TF-IDF + Logistic Regression",
        "MiniLM + Logistic Regression"
    ],
    "Accuracy": [
        accuracy_score(test["label"], m1_test_pred),
        accuracy_score(test["label"], m2_test_pred)
    ],
    "Macro_F1": [
        f1_score(test["label"], m1_test_pred, average="macro"),
        f1_score(test["label"], m2_test_pred, average="macro")
    ]
})

print(results_original.round(4))

results_original.to_csv(
    OUTPUT / "overall_test_results.csv",
    index=False
)

                         Method  Accuracy  Macro_F1
0  TF-IDF + Logistic Regression    0.9780    0.9452
1  MiniLM + Logistic Regression    0.9819    0.9564


*As we notice here, both methods achived a very strong preformance on the original dataset. The first method achived an accuracy of 97% and a macro F1 score of 0.95. On the other hand, the second method preformed slightly better with 98% accuracy and macro F1 score of 0.96*

The results shows that both models are suitable for spam detection. But MiniLM provides a small higher performance on the test set.


### Misspelling Test Evaluation

A modified test dataset was prepared to examine how well the model handles misspellings. About 30% of suitable words in spam messages were modified using character substitutions while keeping their originag meaning. Both models were not trained on these modified messages.


In [10]:
# Misspelled Test Evaluation

m1_miss_pred = model.predict(misspelled_test["text"])

X_test_miss = embed(misspelled_test["text"])
m2_miss_pred = model2.predict(X_test_miss)

results_misspelled = pd.DataFrame({
    "Method": [
        "TF-IDF + Logistic Regression",
        "MiniLM + Logistic Regression"
    ],
    "Accuracy": [
        accuracy_score(misspelled_test["label"], m1_miss_pred),
        accuracy_score(misspelled_test["label"], m2_miss_pred)
    ],
    "Macro_F1": [
        f1_score(misspelled_test["label"], m1_miss_pred, average="macro"),
        f1_score(misspelled_test["label"], m2_miss_pred, average="macro")
    ]
})

print("Misspelled Test Results")
display(results_misspelled.round(4))

Misspelled Test Results


,Method,Accuracy,Macro_F1
0,TF-IDF + Logistic Regression,0.9729,0.9310
1,MiniLM + Logistic Regression,0.9871,0.9694


### Discussion of Misspelling Results

The misspelled test set was used to evaluate how robust each method against misspelling. After adding misspellings, the first model achieved an accuracy of 97% and a macro F1 score of 0.93.While the second model achieved a slightly higher accuracy of 99% and a macro F1 score of 0.97.

Since MiniLM generates contextual sentence embeddings rather than relying on exact word matching, it was less affected by small spelling changes.



In [11]:
comparison = pd.DataFrame({
    "Method": [
        "TF-IDF + LR",
        "MiniLM + LR"
    ],
    "Original Accuracy": results_original["Accuracy"],
    "Misspelled Accuracy": results_misspelled["Accuracy"],
    "Accuracy Drop":
        results_original["Accuracy"] -
        results_misspelled["Accuracy"],
    "Original Macro F1": results_original["Macro_F1"],
    "Misspelled Macro F1": results_misspelled["Macro_F1"],
    "Macro F1 Drop":
        results_original["Macro_F1"] -
        results_misspelled["Macro_F1"]
})

print("Comparison Before and After Misspellings")
display(comparison.round(4))

Comparison Before and After Misspellings


,Method,Original Accuracy,Misspelled Accuracy,Accuracy Drop,Original Macro F1,Misspelled Macro F1,Macro F1 Drop
0,TF-IDF + LR,0.9780,0.9729,0.0052,0.9452,0.9310,0.0143
1,MiniLM + LR,0.9819,0.9871,-0.0052,0.9564,0.9694,-0.0130


### Comparison Before and After Misspellings

As shown here, is the impact of spelling modifications on spam detection performance. For the TF-IDF model, both accuracy and macro F1 decreased after applying misspellings. The decrease of performance is expected. As TF-IDF relies on the exact forms of words, it is sensitive to spelling changes.

On the other hand, the MiniLM model showed a slight impact from the added misspellings. It maintained strong classification performance. And this indicates that semantic sentence embeddings are more robust to variations in word spelling.

As a conclusion, MiniLM with Logistic Regression is more effective method for detecting spam messages that contain misspellings. Therefore, the pretrained embedding approach handles spelling variations more reliably than the traditional TF-IDF approach.

### Error Analysis

The following examples show incorrectly classified messages. These cases help explain the limitations of the models.

In [12]:
errors = test.copy()
errors["prediction"] = m2_test_pred

errors = errors[
    errors["label"] != errors["prediction"]
]

for i, row in errors.head(3).iterrows():
    print("-" * 80)
    print("Message:")
    print(row["text"])
    print()
    print("True Label:", row["label"])
    print("Predicted:", row["prediction"])

--------------------------------------------------------------------------------
Message:
Hi babe its Jordan, how r u? Im home from abroad and lonely, text me back if u wanna chat xxSP visionsms.com Text stop to stopCost 150p 08712400603

True Label: spam
Predicted: ham
--------------------------------------------------------------------------------
Message:
TheMob>Yo yo yo-Here comes a new selection of hot downloads for our members to get for FREE! Just click & open the next link sent to ur fone...

True Label: spam
Predicted: ham
--------------------------------------------------------------------------------
Message:
Xmas & New Years Eve tickets are now on sale from the club, during the day from 10am till 8pm, and on Thurs, Fri & Sat night this week. They're selling fast!

True Label: spam
Predicted: ham


In [13]:
# Number of Spam SMS that are in the test set
spam_mask = test["label"] == "spam"

# Method 1
m1_original_spam_correct = (
    (m1_test_pred[spam_mask] == "spam").sum()
)

m1_modified_spam_correct = (
    (m1_miss_pred[spam_mask] == "spam").sum()
)

m1_spam_to_ham = (
    ((m1_test_pred[spam_mask] == "spam") &
     (m1_miss_pred[spam_mask] == "ham")).sum()
)

# Method 2
m2_original_spam_correct = (
    (m2_test_pred[spam_mask] == "spam").sum()
)

m2_modified_spam_correct = (
    (m2_miss_pred[spam_mask] == "spam").sum()
)

m2_spam_to_ham = (
    ((m2_test_pred[spam_mask] == "spam") &
     (m2_miss_pred[spam_mask] == "ham")).sum()
)

total_spam = spam_mask.sum()

print("Total Spam Messages:", total_spam)
print()

print("TF-IDF + LR")
print("Original cases:", f"{m1_original_spam_correct}/{total_spam}")
print("Modified cases:", f"{m1_modified_spam_correct}/{total_spam}")
print("Spam to Ham:", m1_spam_to_ham)

print()

print("MiniLM + LR")
print("Original cases:", f"{m2_original_spam_correct}/{total_spam}")
print("Modified cases:", f"{m2_modified_spam_correct}/{total_spam}")
print("Spam to Ham:", m2_spam_to_ham)

Total Spam Messages: 96

TF-IDF + LR
Original cases: 79/96
Modified cases: 75/96
Spam to Ham: 5

MiniLM + LR
Original cases: 84/96
Modified cases: 88/96
Spam to Ham: 0


## Try it
Edit the message in the cell and rerun it. The current demo uses Method 1. After the comparison, the team can pass the selected model's prediction function into `try_sms`.

In [14]:
def try_sms(message, predict_label):
    if not message.strip():
        return "Enter a non-empty SMS message."
    label = predict_label(message)
    return {"spam": "Spam", "ham": "Ham"}[label]

def predict_method1(message):
    return model.predict([message])[0]

assert try_sms("   ", predict_method1) == "Enter a non-empty SMS message."
message = "Urgent! You have bein selacted fir a speacial riward. Click the link and confirm your details now."
result = try_sms(message, predict_method1)
assert result in {"Spam", "Ham"}
print(message)
print(result)

Urgent! You have bein selacted fir a speacial riward. Click the link and confirm your details now.
Ham


## Conclusion

The purpose of this project was to compare two approaches for SMS spam detection. And inspecting their ability to handle misspelled spam messages. The results showed that both methods achieved high performance on the original test set.

After adding misspellings into spam messages, the performance of both methods remained strong. However, the MiniLM-based model achieved higher accuracy and macro F1 scores and was less affected misspillings. This shows that pretrained sentence embeddings are more robust to misspellings. Traditional TF-IDF features rely more heavily on exact word matching.

Based on these insights, MiniLM + Logistic Regression was the more effective method. The error analysis also showed that some spam messages can be difficult to identify when they appear as normal conversational or promotional messages. Overall, pretrained language representations can improve spam detection performance, particularly when messages contain spelling variations.

## Sources
- Data: [UCI SMS Spam Collection](https://archive.ics.uci.edu/dataset/228/sms+spam+collection), T. A. Almeida and J. M. Gómez Hidalgo, licensed under CC BY 4.0. If the UCI download fails, an unchanged copy is loaded from the team's [GitHub repository](https://github.com/NabaaNabeeh/sms-spam-misspellings).
- Pretrained model: [all-MiniLM-L6-v2](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2), loaded with the sentence-transformers library (N. Reimers and I. Gurevych, "Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks," EMNLP-IJCNLP, 2019).
- Libraries: pandas, scikit-learn, joblib and sentence-transformers. Colab includes them by default; in Jupyter, install any missing package with `%pip install`. The verified versions are printed in Setup.
